# OralGuard (OC-Detect) — Model Training

**Dataset**: [zaidpy/oral-cancer-dataset](https://www.kaggle.com/datasets/zaidpy/oral-cancer-dataset) (Kaggle)

**Architecture**: EfficientNet-B4 with MC Dropout

**Features**: CLAHE augmentation • Grad-CAM explainability • Uncertainty quantification

**Classification**: Binary — `CANCER` vs `NON CANCER`

## 0. Install & Download Dataset

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_1590b785fb11f0b8186146787c9c897c'

import kagglehub
dataset_path = kagglehub.dataset_download('zaidpy/oral-cancer-dataset')
print('Dataset downloaded to:', dataset_path)

## 1. Imports & Configuration

In [ ]:
import os, cv2, glob, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

CONFIG = {
    'NUM_CLASSES': 2,
    'CLASS_NAMES': ['NON CANCER', 'CANCER'],
    'IMAGE_SIZE': 380,
    'BATCH_SIZE': 16,
    'EPOCHS': 20,
    'LR': 1e-4,
    'WEIGHT_DECAY': 1e-5,
    'MC_DROPOUT_T': 30,
    'MODEL_SAVE_DIR': '../models',
}

## 2. Load & Merge Dataset

In [ ]:
# Collect images from both dataset versions
image_paths, labels = [], []
exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')

for root, dirs, files in os.walk(dataset_path):
    folder_name = os.path.basename(root).upper()
    if folder_name == 'CANCER':
        label = 1
    elif folder_name == 'NON CANCER':
        label = 0
    else:
        continue
    for ext in exts:
        for f in glob.glob(os.path.join(root, ext)):
            image_paths.append(f)
            labels.append(label)

print(f'Total images: {len(image_paths)}')
print(f'  CANCER: {labels.count(1)}, NON CANCER: {labels.count(0)}')

# Stratified train/val/test split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print(f'\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## 3. Dataset & Augmentation (with CLAHE)

In [ ]:
train_transform = A.Compose([
    A.Resize(CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE']),
    A.CLAHE(clip_limit=2.0, p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=25, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(p=0.2),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE']),
    A.CLAHE(clip_limit=2.0, p=1.0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class OralDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        if img is None:
            img = np.zeros((CONFIG['IMAGE_SIZE'], CONFIG['IMAGE_SIZE'], 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, self.labels[idx]

train_ds = OralDataset(X_train, y_train, train_transform)
val_ds   = OralDataset(X_val, y_val, val_transform)
test_ds  = OralDataset(X_test, y_test, val_transform)

# Weighted sampler for class imbalance
class_counts = np.bincount(y_train)
weights = 1.0 / class_counts
sample_weights = [weights[l] for l in y_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH_SIZE'], sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=0)

print(f'Loaders ready — Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

## 4. Visualize Samples

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

for i, ax in enumerate(axes.flat):
    img, label = train_ds[i]
    img_np = img.permute(1, 2, 0).numpy() * std + mean
    img_np = np.clip(img_np, 0, 1)
    ax.imshow(img_np)
    ax.set_title(CONFIG['CLASS_NAMES'][label], fontsize=12,
                 color='red' if label == 1 else 'green')
    ax.axis('off')
plt.suptitle('Training Samples (with CLAHE)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Model — EfficientNet-B4 + MC Dropout

In [ ]:
class OralClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

model = OralClassifier(CONFIG['NUM_CLASSES']).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['LR'], weight_decay=CONFIG['WEIGHT_DECAY'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['EPOCHS'])

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total_params:,} total, {trainable:,} trainable')

## 6. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, desc='Val', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return loss_sum / total, correct / total

In [ ]:
os.makedirs(CONFIG['MODEL_SAVE_DIR'], exist_ok=True)
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, CONFIG['EPOCHS'] + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'config': CONFIG
        }, os.path.join(CONFIG['MODEL_SAVE_DIR'], 'best_oral_cancer_model.pth'))
        marker = ' ✓ BEST'

    print(f'Epoch {epoch:02d}/{CONFIG["EPOCHS"]} │ '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} │ '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}{marker}')

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

## 7. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train')
ax1.plot(epochs_range, history['val_loss'], 's-', label='Val')
ax1.set_title('Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'o-', label='Train')
ax2.plot(epochs_range, history['val_acc'], 's-', label='Val')
ax2.set_title('Accuracy', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['MODEL_SAVE_DIR'], 'training_curves.png'), dpi=150)
plt.show()

## 8. Test Evaluation

In [ ]:
# Load best model
ckpt = torch.load(os.path.join(CONFIG['MODEL_SAVE_DIR'], 'best_oral_cancer_model.pth'), map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Testing'):
        imgs = imgs.to(DEVICE)
        out = model(imgs)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print('\n' + '='*60)
print('CLASSIFICATION REPORT')
print('='*60)
print(classification_report(all_labels, all_preds, target_names=CONFIG['CLASS_NAMES']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CONFIG['CLASS_NAMES'],
            yticklabels=CONFIG['CLASS_NAMES'], ax=ax1)
ax1.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.4f}')
ax2.plot([0, 1], [0, 1], 'k--', lw=1)
ax2.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR'); ax2.legend(fontsize=12); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['MODEL_SAVE_DIR'], 'evaluation_results.png'), dpi=150)
plt.show()

## 9. Grad-CAM Explainability (XAI-01)

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(1).item()
        self.model.zero_grad()
        output[0, target_class].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = torch.nn.functional.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear')
        return cam.squeeze().cpu().numpy(), target_class

# Visualize Grad-CAM on test samples
gradcam = GradCAM(model, model.backbone.features[-1])
mean_t = np.array([0.485, 0.456, 0.406])
std_t  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i in range(4):
    img_t, label = test_ds[i]
    heatmap, pred_cls = gradcam.generate(img_t.unsqueeze(0).to(DEVICE))

    img_np = img_t.permute(1, 2, 0).numpy() * std_t + mean_t
    img_np = np.clip(img_np, 0, 1)

    axes[0, i].imshow(img_np)
    axes[0, i].set_title(f'True: {CONFIG["CLASS_NAMES"][label]}', fontsize=11)
    axes[0, i].axis('off')

    axes[1, i].imshow(img_np)
    axes[1, i].imshow(heatmap, cmap='jet', alpha=0.4)
    axes[1, i].set_title(f'Pred: {CONFIG["CLASS_NAMES"][pred_cls]}', fontsize=11)
    axes[1, i].axis('off')

plt.suptitle('Grad-CAM Explainability (40% alpha overlay)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['MODEL_SAVE_DIR'], 'gradcam_results.png'), dpi=150)
plt.show()

## 10. MC Dropout Uncertainty (XAI-02)

In [ ]:
def mc_dropout_predict(model, input_tensor, T=30):
    """Run T stochastic forward passes with dropout enabled."""
    model.train()  # Enable dropout
    preds = []
    with torch.no_grad():
        for _ in range(T):
            out = model(input_tensor.to(DEVICE))
            prob = torch.softmax(out, dim=1).cpu().numpy()
            preds.append(prob)
    preds = np.stack(preds)
    mean_prob = preds.mean(axis=0)[0]
    std_prob  = preds.std(axis=0)[0]
    entropy   = -np.sum(mean_prob * np.log(mean_prob + 1e-10))
    return mean_prob, std_prob, entropy

# Demo on test samples
print(f'{"Sample":>8} {"True":>12} {"Predicted":>12} {"Confidence":>12} {"Entropy":>10} {"Uncertainty":>12}')
print('-' * 72)
for i in range(10):
    img_t, label = test_ds[i]
    mean_prob, std_prob, entropy = mc_dropout_predict(model, img_t.unsqueeze(0), T=CONFIG['MC_DROPOUT_T'])
    pred = np.argmax(mean_prob)
    conf = mean_prob[pred]
    unc  = 'HIGH' if entropy > 0.5 else 'LOW'
    print(f'{i:>8} {CONFIG["CLASS_NAMES"][label]:>12} {CONFIG["CLASS_NAMES"][pred]:>12} {conf:>11.4f} {entropy:>10.4f} {unc:>12}')

## 11. Export for Inference Engine

In [ ]:
# Save final model for OC-Detect inference pipeline
final_path = os.path.join(CONFIG['MODEL_SAVE_DIR'], 'oralguard_efficientnet_b4.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'num_classes': CONFIG['NUM_CLASSES'],
    'class_names': CONFIG['CLASS_NAMES'],
    'image_size': CONFIG['IMAGE_SIZE'],
    'architecture': 'efficientnet_b4',
    'best_val_acc': best_val_acc,
}, final_path)

size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f'Model saved to: {final_path}')
print(f'Model size: {size_mb:.1f} MB')
print(f'Best validation accuracy: {best_val_acc:.4f}')
print('\n✅ Ready for OC-Detect inference engine integration!')